In [15]:
from datetime import datetime, timedelta
import os
import random

import datafusion
from datafusion import SessionContext, col
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake, QueryBuilder

In [ ]:
# Connection settings for Azurite
# (local storage like Azure Blob Storage)
STORAGE_OPTIONS = {
    "account_name": "devstoreaccount1",
    "access_key": (
        "Eby8vdM02xNOcqFlqUwJPLlmEtlCDXJ1OUzFT50uSRZ6IFsuFq2UVErCz4I6tq/"
        "K1SZFPTOtr/KBHBeksoGMGw=="
    ),
    "endpoint": "http://azurite:10000/devstoreaccount1",
    "allow_http": "true",
}

# Path in Azurite ('dados-delta' container and 'tabela_vendas' folder)
DELTA_PATH = "az://lakehouse/sales_table"
PARQUET_PATH = "az://lakehouse/sales.parquet"

In [ ]:
# Initialize the DataFusion context (Rust-based engine)
ctx = SessionContext()

# Create the DataFusion DataFrame from the in-memory PyArrow table
df_sales = ctx.from_arrow(sales_table)

# Register the DataFrame as a view (virtual table) for SQL queries
ctx.register_view("sales", df_sales)

print("Table successfully registered in DataFusion!")

Tabela registrada no DataFusion com sucesso!


In [ ]:
# Simple query with filter and aggregation by category
df_sql_1 = ctx.sql("""
    SELECT 
        category,
        COUNT(transaction_id) AS total_transactions,
        SUM(quantity * unit_price) AS total_revenue,
        AVG(unit_price) AS average_price
    FROM sales
    GROUP BY category
    ORDER BY total_revenue DESC
""")
df_sql_1.show()

DataFrame()
+-------------+------------------+--------------------+-------------------+
| categoria   | total_transacoes | faturamento_total  | media_preco       |
+-------------+------------------+--------------------+-------------------+
| Roupas      | 5045             | 11654618.560000012 | 762.7396075322105 |
| Alimentos   | 5061             | 11485936.869999997 | 758.961331752618  |
| Eletrônicos | 5034             | 11391821.890000008 | 757.1748132697661 |
| Esportes    | 4966             | 11328225.719999991 | 766.1628513894487 |
| Casa        | 4894             | 11021495.320000006 | 765.8510850020436 |
+-------------+------------------+--------------------+-------------------+


In [ ]:
# Using Window Functions
df_sql_2 = ctx.sql("""
    WITH ranked_sales AS (
        SELECT 
            city,
            category,
            unit_price,
            ROW_NUMBER() OVER (
                PARTITION BY city 
                ORDER BY unit_price DESC
            ) AS rk
        FROM sales
    )
    SELECT city, category, unit_price
    FROM ranked_sales
    WHERE rk <= 2
""")
df_sql_2.show()

DataFrame()
+----------------+-------------+----------------+
| cidade         | categoria   | preco_unitario |
+----------------+-------------+----------------+
| São Paulo      | Casa        | 1499.62        |
| São Paulo      | Roupas      | 1498.08        |
| Porto Alegre   | Esportes    | 1499.95        |
| Porto Alegre   | Eletrônicos | 1499.69        |
| Rio de Janeiro | Roupas      | 1499.63        |
| Rio de Janeiro | Esportes    | 1499.26        |
| Curitiba       | Esportes    | 1499.97        |
| Curitiba       | Alimentos   | 1499.7         |
| Belo Horizonte | Casa        | 1499.76        |
| Belo Horizonte | Roupas      | 1499.17        |
+----------------+-------------+----------------+


In [ ]:
# Selecting columns, creating a new calculated column, and filtering
df_expr = (
    ctx.table("sales")
    .filter(col("quantity") >= 3)
    .select(
        col("transaction_id"),
        col("city"),
        col("category"),
        (col("quantity") * col("unit_price")).alias("total_amount"),
    )
    .sort(col("total_amount").sort(ascending=False))
    .limit(5)
)

df_expr.show()

DataFrame()
+--------------+----------------+-----------+-------------------+
| id_transacao | cidade         | categoria | valor_total       |
+--------------+----------------+-----------+-------------------+
| 1342         | São Paulo      | Casa      | 7498.099999999999 |
| 24751        | Curitiba       | Casa      | 7496.45           |
| 3062         | Rio de Janeiro | Esportes  | 7496.3            |
| 19288        | Porto Alegre   | Alimentos | 7494.6            |
| 12461        | Belo Horizonte | Roupas    | 7491.549999999999 |
+--------------+----------------+-----------+-------------------+


In [ ]:
write_deltalake(
    DELTA_PATH,
    sales_table,
    storage_options=STORAGE_OPTIONS,
    mode="overwrite",
)

In [ ]:
dt = DeltaTable(DELTA_PATH, storage_options=STORAGE_OPTIONS)

# 2. Executes full SELECT * in the Rust engine (pulls everything into memory as Arrow)
result_arrow = (
    QueryBuilder()
    .register("sales_delta", dt)
    .execute("""
        SELECT * 
        FROM sales_delta
    """)
    .read_all()
)

# Displays only a formatted preview (initial sample)
print(result_arrow.slice(0, 5))

Sucesso! Total de linhas carregadas: 25000
Total de colunas: 6

arro3.core.Table
+--------------+-------------+----------------+------------+----------------+---------------------+
| id_transacao | categoria   | cidade         | quantidade | preco_unitario | data_venda          |
| Int64        | Utf8View    | Utf8View       | Int64      | Float64        | Utf8View            |
+--------------+-------------+----------------+------------+----------------+---------------------+
| 1            | Eletrônicos | Belo Horizonte | 2          | 1141.49        | 2026-08-26 02:04:32 |
| 2            | Eletrônicos | Curitiba       | 1          | 1392.07        | 2026-09-15 15:04:32 |
| 3            | Casa        | Curitiba       | 5          | 1006.08        | 2026-09-20 15:04:32 |
| 4            | Roupas      | Porto Alegre   | 5          | 632.0          | 2026-09-17 07:04:32 |
| 5            | Roupas      | São Paulo      | 3          | 1074.49        | 2026-09-09 17:04:32 |
+--------------+---